GravNet: Qkeras vs hls4ml
=============================

In this Jupyter notebook, we take a trained Keras GravNet model, convert it to hls using hls4ml and evaluate accuracies.

In [ ]:
from qgravnet import QGravNetFactory

from hls4ml_gravnet.utils.data import load_processed
from hls4ml_gravnet.utils.evaluation import load_run
from hls4ml_gravnet.utils.files import DATASET_PATH, RESULTS_PATH

train_dir = RESULTS_PATH / 'Jan21_128vertex'

model_cfg, weights_path, history, datapath, n_vertices, _ = load_run(train_dir=train_dir)
datapath = DATASET_PATH / 'toy_calo/toy_calo_processed.h5' # Reduced dataset

D = load_processed(datapath)

trained_model = QGravNetFactory(**model_cfg).create_keras_model(n_vertices, 4)
trained_model.load_weights(weights_path)


trained_model.summary()

In [ ]:
from hls4ml_gravnet.hls4ml_extension.register_extensions import register_extensions

try:
    register_extensions(backend='Vitis')
except Exception:
    pass  # Already registered

In [ ]:
import hls4ml
from hls4ml_gravnet.utils.files import HLS4ML_OUT_PATH
from hls4ml_gravnet.utils.hls_config import set_qgravnet_hls_config

hls_config = hls4ml.utils.config_from_keras_model(trained_model, granularity='name', backend='Vitis', default_reuse_factor=1)
set_qgravnet_hls_config(hls_config)

for layer in hls_config['LayerName'].keys():
    hls_config['LayerName'][layer]['Trace'] = True

hls_model = hls4ml.converters.convert_from_keras_model(
    trained_model, hls_config=hls_config, output_dir=str(HLS4ML_OUT_PATH / 'toy_calo'), backend='Vitis'
)
print(hls_config)

In [ ]:
test_energy_pred, test_pid_pred = trained_model.predict(D['X_hits_test'])
hls_pred, hls_trace = hls_model.trace(D['X_hits_test'])
hls_test_energy_pred, hls_test_pid_pred = hls_pred[0], hls_pred[1]

In [ ]:
from hls4ml_gravnet.utils.evaluation import compare_keras_hls_predictions

compare_keras_hls_predictions(
    test_energy_pred=test_energy_pred,
    test_pid_pred=test_pid_pred,
    test_energy_pred_hls=hls_test_energy_pred,
    test_pid_pred_hls=hls_test_pid_pred,
    test_energy_true=D['y_energy_test'],
    test_pid_true=D['y_pid_test'],
)

## Tracing & Profiling

Analyze step sizes across the energy range. The result should be a horizontally oriented cloud of points, indicating that step sizes are not affected by quantization and large energies can be predicted with the same granularity as low energies.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def analyze_steps_small_data(hls_pred_linear):
    unique_preds = np.unique(hls_pred_linear)
    steps = np.diff(unique_preds)

    plt.figure(figsize=(8, 4))
    plt.scatter(unique_preds[:-1], steps, alpha=0.6, s=10)
    plt.xlabel('Predicted Energy (GeV)')
    plt.ylabel('Step Size (GeV)')
    plt.title('Quantization Step Size (Visible on Small Data)')
    plt.grid(True, which='both', linestyle='--')
    plt.semilogy()  # Log scale helps see the steps grow
    plt.show()


analyze_steps_small_data(hls_test_energy_pred.flatten())

In [ ]:
from hls4ml.model.profiling import get_ymodel_keras

keras_trace = get_ymodel_keras(trained_model, D['X_hits_test'])

Compare min / max outputs of layers for a quick overview of ranges.

In [ ]:
import numpy as np

for key, value in keras_trace.items():
    print(f'{key}: {np.max(value)}, {np.min(value)}')

print()
for key, value in hls_trace.items():
    print(f'{key}: {np.max(value)}, {np.min(value)}')

Check which of the layers that share their name across Keras and HLS have the most mismatches (Keras vs HLS).

In [ ]:
import numpy as np

ATOL = 0.5

common_layers = [key for key in hls_trace.keys() if key in keras_trace]
print(f'Checking {len(common_layers)} layers for mismatches...')

for layer in common_layers:
    k_val = keras_trace[layer]
    h_val = hls_trace[layer]

    diff = np.abs(h_val - k_val)

    if not np.allclose(k_val, h_val, rtol=0, atol=ATOL):
        bad_indices = np.where(diff > ATOL)
        max_diff = np.max(diff)
        mean_diff = np.mean(diff)

        print(f'\n--- MISMATCH FOUND IN LAYER: {layer} ---')
        print(f'  Max absolute error: {max_diff:.4f}')
        print(f'  Mean absolute error: {mean_diff:.4f}')
        print(f'  Total mismatch count: {len(bad_indices[0])} / {diff.size}')

        print('  First 5 mismatches:')
        for i in range(min(len(bad_indices[0]), 5)):
            idx = tuple(bad_indices[dim][i] for dim in range(len(bad_indices)))

            k_elem = k_val[idx]
            h_elem = h_val[idx]
            d_elem = diff[idx]
            print(f'    Index {idx}: Keras={k_elem:.4f}, HLS={h_elem:.4f}, Diff={d_elem:.4f}')

Analyze if there is any bias propagated through the model by plotting mean difference and standard deviation for each layer output.

In [ ]:
layer_names = []
biases = []
errors = []

for layer in common_layers:
    k_val = keras_trace[layer].flatten()
    h_val = hls_trace[layer].flatten()

    diff = h_val - k_val
    bias = np.mean(diff)
    std_dev = np.std(diff)

    layer_names.append(layer)
    biases.append(bias)
    errors.append(std_dev)

plt.figure(figsize=(14, 6))
plt.errorbar(layer_names, biases, yerr=errors, fmt='o-', capsize=5, ecolor='gray', label='Mean Bias ± 1 Std Dev')
plt.axhline(0, color='red', linestyle='--', linewidth=1, label='Zero Bias')
plt.xticks(rotation=90)  # Rotate layer names for readability
plt.ylabel('HLS - Keras Difference')
plt.title('Layer-wise Output Bias Propagation')
plt.legend()
plt.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()

Weight accuracy distributions

In [ ]:
from hls4ml.model.profiling import numerical

numerical(model=trained_model, hls_model=hls_model)

Compare energy prediction distributions (Keras vs HLS vs Ground Truth).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(test_energy_pred, bins=30, alpha=0.6, label='Keras', color='skyblue', edgecolor='black')
plt.hist(hls_test_energy_pred, bins=30, alpha=0.6, label='HLS', color='salmon', edgecolor='black')
plt.hist(D['y_energy_test'], bins=30, alpha=0.6, label='Ground Truth', color='lightgreen', edgecolor='black')
plt.title('Histogram Comparison Energy Predictions')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.3)

plt.show()